In [ ]:
# =============================================================================
# Author: G.A.N.M.Perera
# Student ID: 20240694
# Peer Reviewer: [Enter Peer Name Here]
# Review Date: 15/04/2026
# Module: 5DATA002W.2 — Machine Learning & Data Mining
# Task: Notebook 3
# =============================================================================

In [1]:
# Bring in pandas for tabular data processing
import pandas as pd

# Read the cleaned loan dataset into a DataFrame
df = pd.read_csv('/content/cleaned_loan_data (1).csv')

# Preview the top 5 records to ensure successful loading
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/cleaned_loan_data (1).csv'

In [ ]:
# Cast the dataset to object type to manage categorical features uniformly
df = df.astype('object')

# Force the classification target back to integer format
df['loan_approval_status'] = df['loan_approval_status'].astype(int)

# Isolate predictor variables by dropping the target and the non-informative ID column
X = df.drop(columns=['id', 'loan_approval_status'])

# Isolate the target label for the classification task
y = df['loan_approval_status']

from sklearn.model_selection import train_test_split

# Create a 70/30 train-test split, maintaining class distribution via stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

# Build a pipeline that first fills missing values with column means,
# then feeds the completed data into a Logistic Regression classifier.
# This sequence prevents data leakage during the imputation step.
model = make_pipeline(SimpleImputer(strategy='mean'), LogisticRegression(max_iter=1000))

# Train the pipeline on the training split
model.fit(X_train, y_train)

# Generate predictions on the unseen test data
y_pred_lr = model.predict(X_test)

# Output the performance metrics (Precision, Recall, F1-Score)
print(classification_report(y_test, y_pred_lr))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Compute the test confusion matrix for the Logistic Regression base learner
cm_lr = confusion_matrix(y_test, y_pred_lr)

# Create a visual display of the confusion matrix with labelled axes
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['Not Approved (0)', 'Approved (1)'])

# Plot the confusion matrix heatmap using a blue colour map
disp_lr.plot(cmap='Blues')

# Add a descriptive title to the plot
plt.title('Logistic Regression - Test Confusion Matrix')

# Render the plot
plt.show()

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

# Construct a pipeline for a Naive Bayes classifier, preceded by mean imputation.
# GaussianNB is selected for its ability to process negative feature values effectively.
nb_model = make_pipeline(SimpleImputer(strategy='mean'), GaussianNB())

# Fit the Naive Bayes pipeline to the training data
nb_model.fit(X_train, y_train)

# Evaluate the model on the test set
y_pred_nb = nb_model.predict(X_test)

# Display the resulting classification report
print(classification_report(y_test, y_pred_nb))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Compute the test confusion matrix for the Naive Bayes base learner
cm_nb = confusion_matrix(y_test, y_pred_nb)

# Create a visual display of the confusion matrix with labelled axes
disp_nb = ConfusionMatrixDisplay(confusion_matrix=cm_nb, display_labels=['Not Approved (0)', 'Approved (1)'])

# Plot the confusion matrix heatmap using a blue colour map
disp_nb.plot(cmap='Blues')

# Add a descriptive title to the plot
plt.title('Naive Bayes - Test Confusion Matrix')

# Render the plot
plt.show()

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report

# Group our previously built pipelines (Logistic Regression & Gaussian NB)
# to act as foundational models for the ensemble.
base_learners = [('LR', model), ('NB', nb_model)]

# Initialize a Voting Classifier that aggregates the predicted probabilities ('soft' voting)
ensemble = VotingClassifier(estimators=base_learners, voting='soft')

# Train the ensemble setup.
# Missing values are inherently handled by the SimpleImputer inside each base learner's pipeline.
ensemble.fit(X_train, y_train)

# Retrieve predictions from the combined model
y_pred_ensemble = ensemble.predict(X_test)

# Check if the ensemble approach improved overall classification performance
print(classification_report(y_test, y_pred_ensemble))


In [ ]:
from sklearn.metrics import RocCurveDisplay

# Visualize the model's diagnostic ability by plotting the ROC curve for the ensemble model
RocCurveDisplay.from_estimator(ensemble, X_test, y_test)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Shift to a regression task: estimating the continuous 'loan_amount' variable.
# Remove the ID, original classification target, and the new regression target from the features.
X_reg = df.drop(columns=['id', 'loan_amount', 'loan_approval_status'])
y_reg = df['loan_amount']

# Partition the regression dataset, ensuring reproducible results with a fixed random state
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

# Impute missing feature values with the column average before fitting the regression models
imputer = SimpleImputer(strategy='mean')
X_train_reg_imputed = imputer.fit_transform(X_train_reg)

# Initialize a standard Decision Tree without depth restrictions
full_tree = DecisionTreeRegressor(random_state=42)

# Initialize a constrained Decision Tree to mitigate overfitting risks
pruned_tree = DecisionTreeRegressor(max_depth=5, min_samples_leaf=10, random_state=42)

# Fit both regression models on the preprocessed training data
full_tree.fit(X_train_reg_imputed, y_train_reg)
pruned_tree.fit(X_train_reg_imputed, y_train_reg)

print('Regression models trained successfully.')